# EYE-D 이미지 데이터 뷰어

이 노트북은 data/filtered 디렉토리 내의 정제된 크롭 이미지들과 data/market1501 포맷 변환 완료된 데이터를 대화형(Interactive) 위젯을 통해 편리하게 시각화하여 조회하는 도구입니다.

In [ ]:
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# 기본 데이터 경로
FILTERED_DIR = Path("../data/filtered")
MARKET_DIR = Path("../data/market1501")

# 노트북이 위치한 경로 기준 예외 처리
if not FILTERED_DIR.exists() and Path("data/filtered").exists():
    FILTERED_DIR = Path("data/filtered")
    MARKET_DIR = Path("data/market1501")

print(f"[INFO] Filtered 데이터 경로: {FILTERED_DIR.resolve()}")
print(f"[INFO] Market1501 데이터 경로: {MARKET_DIR.resolve()}")

## 1. Filtered Tracklets 뷰어
data/filtered 하위의 트랙렛별 이미지들을 조회합니다. 글로벌 ID 병합 상태를 한눈에 볼 수 있습니다.

In [ ]:
# 1. 트랙렛 목록 및 글로벌 ID 정보 수집
tracklet_dirs = sorted([p for p in FILTERED_DIR.glob("c*_*[0-9]/track_*") if p.is_dir()])
tracklet_options = []

for tdir in tracklet_dirs:
    meta_path = tdir / "metadata.json"
    global_id = "N/A"
    if meta_path.exists():
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
                global_id = meta.get("global_id", "N/A")
        except:
            pass
    label = f"{tdir.parent.name}/{tdir.name} (Global ID: {global_id})"
    tracklet_options.append((label, tdir))

if not tracklet_options:
    print("[WARN] data/filtered 디렉토리에 트랙렛이 존재하지 않습니다.")
else:
    tracklet_select = widgets.Dropdown(
        options=tracklet_options,
        description='트랙렛 폴더:',
        layout={'width': '450px'}
    )

    max_cols_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=16,
        step=1,
        description='열(Cols):'
    )

    out_filtered = widgets.Output()

    def show_filtered_images(change=None):
        tdir = tracklet_select.value
        cols = max_cols_slider.value
        
        with out_filtered:
            clear_output(wait=True)
            if not tdir or not tdir.exists():
                print("선택한 폴더가 유효하지 않습니다.")
                return
                
            images = sorted(list(tdir.glob("*.jpg")))
            if not images:
                print("해당 트랙렛 디렉토리에 JPG 이미지가 없습니다.")
                return
                
            print(f"경로: {tdir.parent.name}/{tdir.name} | 이미지 수: {len(images)}장")
            
            rows = int(np.ceil(len(images) / cols))
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.8))
            
            if len(images) == 1:
                axes = np.array([axes])
            axes = axes.flatten()
            
            for idx, img_path in enumerate(images):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(img_path.name, fontsize=7)
                axes[idx].axis('off')
                
            for idx in range(len(images), len(axes)):
                axes[idx].axis('off')
                
            plt.tight_layout()
            plt.show()

    tracklet_select.observe(show_filtered_images, names='value')
    max_cols_slider.observe(show_filtered_images, names='value')

    display(widgets.VBox([tracklet_select, max_cols_slider, out_filtered]))
    show_filtered_images()

## 2. Market-1501 변환 데이터 뷰어
data/market1501 하위의 bounding_box_train, bounding_box_test, query 폴더의 이미지를 고유 ID별로 필터링하여 조회합니다.

In [ ]:
if not MARKET_DIR.exists():
    print("[WARN] Market-1501 데이터셋이 존재하지 않습니다. 먼저 scripts/format_market1501.py를 구동하세요.")
else:
    split_select = widgets.Dropdown(
        options=['bounding_box_train', 'bounding_box_test', 'query'],
        value='bounding_box_train',
        description='데이터 분할:'
    )

    id_select = widgets.Dropdown(
        options=[],
        description='고유 ID 선택:',
        layout={'width': '250px'}
    )

    out_market = widgets.Output()

    def update_ids(*args):
        split_dir = MARKET_DIR / split_select.value
        if not split_dir.exists():
            id_select.options = []
            return
            
        images = split_dir.glob("*.jpg")
        ids = set()
        for img_path in images:
            parts = img_path.name.split('_')
            if parts:
                ids.add(parts[0])
                
        sorted_ids = sorted(list(ids))
        id_select.options = sorted_ids

    def show_market_images(change=None):
        split_dir = MARKET_DIR / split_select.value
        target_id = id_select.value
        
        with out_market:
            clear_output(wait=True)
            if not split_dir.exists():
                print(f"'{split_select.value}' 디렉토리가 없습니다.")
                return
            if not target_id:
                print("선택된 고유 인물 ID가 없습니다.")
                return
                
            images = sorted(list(split_dir.glob(f"{target_id}_*.jpg")))
            print(f"고유 ID: {target_id} | 검색된 이미지 수: {len(images)}장")
            
            # 이미지 시각화 개수 최대 제한 (과도한 부하 차단)
            limit = 48
            if len(images) > limit:
                print(f"(속도 저하 방지를 위해 상위 {limit}장만 미리보기합니다.)")
                images = images[:limit]
                
            cols = 8
            rows = int(np.ceil(len(images) / cols))
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.8))
            
            if len(images) == 1:
                axes = np.array([axes])
            axes = axes.flatten()
            
            for idx, img_path in enumerate(images):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                # 파일명 속 카메라/슬롯 정보 파싱
                parts = img_path.name.split('_')
                cam_slot = parts[1] if len(parts) > 1 else ""
                frame = parts[2] if len(parts) > 2 else ""
                axes[idx].set_title(f"{cam_slot}\n{frame}", fontsize=7)
                axes[idx].axis('off')
                
            for idx in range(len(images), len(axes)):
                axes[idx].axis('off')
                
            plt.tight_layout()
            plt.show()

    split_select.observe(update_ids, names='value')
    id_select.observe(show_market_images, names='value')

    update_ids()
    display(widgets.VBox([split_select, id_select, out_market]))
    
    if id_select.options:
        id_select.value = id_select.options[0]
        show_market_images()